# Next Word Predictor Using LSTM - PyTorch Version
This notebook contains the PyTorch implementation of an LSTM model for next word prediction.
The model learns to predict the next word in a sequence.

In [ ]:
# Sample text data - FAQs about Data Science Program
faqs = """About the Program
What is the course fee for Data Science Mentorship Program (DSMP 2023)
The course follows a monthly subscription model where you have to make monthly payments of Rs 799/month.
What is the total duration of the course?
The total duration of the course is 7 months. So the total course fee becomes 799*7 = Rs 5600(approx.)
What is the syllabus of the mentorship program?
We will be covering the following modules:
Python Fundamentals
Python libraries for Data Science
Data Analysis
SQL for Data Science
Maths for Machine Learning
ML Algorithms
Practical ML
MLOPs
Case studies
Will Deep Learning and NLP be a part of this program?
No, NLP and Deep Learning both are not a part of this program's curriculum.
"""

print(f"Text data loaded. Length: {len(faqs)} characters")

In [ ]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import time
from tqdm import tqdm

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Create a Tokenizer for the text
class TextTokenizer:
    def __init__(self):
        self.word_index = {}
        self.index_word = {}
        self.vocab_size = 0
    
    def fit_on_text(self, text):
        """Build vocabulary from text"""
        words = text.lower().split()
        unique_words = sorted(set(words))
        
        self.word_index = {word: idx for idx, word in enumerate(unique_words)}
        self.index_word = {idx: word for word, idx in self.word_index.items()}
        self.vocab_size = len(self.word_index)
    
    def text_to_sequences(self, text):
        """Convert text to sequence of integers"""
        words = text.lower().split()
        return [self.word_index[word] for word in words]
    
    def sequences_to_text(self, sequences):
        """Convert sequences of integers back to text"""
        return ' '.join([self.index_word[idx] for idx in sequences])


# Initialize and fit tokenizer
tokenizer = TextTokenizer()
tokenizer.fit_on_text(faqs)

print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Sample word_index entries: {dict(list(tokenizer.word_index.items())[:10])}")

In [ ]:
# Convert text to sequences
sequence = tokenizer.text_to_sequences(faqs)
print(f"Total sequence length: {len(sequence)}")
print(f"First 20 tokens: {sequence[:20]}")

In [ ]:
# Create input-output pairs for training
def create_sequences(sequence, seq_length):
    """
    Create input-output pairs where:
    Input: seq_length tokens
    Output: next token (for prediction)
    """
    X = []
    y = []
    
    for i in range(len(sequence) - seq_length):
        X.append(sequence[i:i + seq_length])
        y.append(sequence[i + seq_length])
    
    return np.array(X), np.array(y)


seq_length = 5  # Context window size
X, y = create_sequences(sequence, seq_length)

print(f"Input sequences shape: {X.shape}")
print(f"Output labels shape: {y.shape}")
print(f"Sample input sequence: {X[0]} -> next word index: {y[0]}")
print(f"Sample input text: {tokenizer.sequences_to_text(X[0])} -> next word: {tokenizer.index_word[y[0]]}")

In [ ]:
# Pad sequences to ensure same length
def pad_sequences_numpy(X, maxlen=None, padding='pre', pad_value=0):
    """Pad sequences"""
    if maxlen is None:
        maxlen = max([len(seq) for seq in X]) if len(X) > 0 else 0
    
    padded = []
    for seq in X:
        if padding == 'pre':
            padded_seq = [pad_value] * (maxlen - len(seq)) + seq
        else:  # 'post'
            padded_seq = seq + [pad_value] * (maxlen - len(seq))
        padded.append(padded_seq)
    
    return np.array(padded), maxlen


# Pad the sequences
X_padded, max_len = pad_sequences_numpy(X, padding='pre')

print(f"Padded input sequences shape: {X_padded.shape}")
print(f"Max sequence length: {max_len}")

In [ ]:
# Create PyTorch Dataset
class NextWordDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).long()
        self.y = torch.from_numpy(y).long()
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Create dataset and dataloader
dataset = NextWordDataset(X_padded, y)
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"Number of batches: {len(dataloader)}")

In [ ]:
# Define the LSTM model for next word prediction
class NextWordLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers=1):
        super(NextWordLSTM, self).__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # LSTM layer
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.5 if num_layers > 1 else 0
        )
        
        # Dense layers
        self.fc1 = nn.Linear(hidden_size, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, vocab_size)
    
    def forward(self, x):
        # Embedding
        x = self.embedding(x)
        
        # LSTM
        lstm_out, (hidden, cell) = self.lstm(x)
        
        # Use the last output
        last_output = lstm_out[:, -1, :]
        
        # Dense layers
        x = self.fc1(last_output)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x


# Initialize model
vocab_size = tokenizer.vocab_size
embedding_dim = 128
hidden_size = 150
num_layers = 1

model = NextWordLSTM(vocab_size, embedding_dim, hidden_size, num_layers)
model = model.to(device)

print(f"Model initialized:")
print(model)

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam")

In [ ]:
# Training function
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for X_batch, y_batch in tqdm(dataloader, desc="Training"):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        # Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    
    return avg_loss, accuracy


print("Training function defined!")

In [ ]:
# Train the model
num_epochs = 20
history = {
    'loss': [],
    'accuracy': []
}

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    
    train_loss, train_acc = train_epoch(model, dataloader, criterion, optimizer, device)
    
    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    
    print(f"Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")

print("\nTraining completed!")

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['loss'])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history['accuracy'])
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training Accuracy')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Function to predict next words
def predict_next_words(seed_text, model, tokenizer, num_words=10, max_seq_length=5):
    """
    Predict the next num_words words given seed text
    """
    generated_text = seed_text
    
    model.eval()
    with torch.no_grad():
        for _ in range(num_words):
            # Tokenize current text
            tokens = tokenizer.text_to_sequences(generated_text)
            
            # Keep only the last max_seq_length tokens
            if len(tokens) > max_seq_length:
                tokens = tokens[-max_seq_length:]
            
            # Pad if necessary
            if len(tokens) < max_seq_length:
                tokens = [0] * (max_seq_length - len(tokens)) + tokens
            
            # Convert to tensor
            input_tensor = torch.tensor([tokens]).long().to(device)
            
            # Predict
            output = model(input_tensor)
            predicted_idx = torch.argmax(output, dim=1).item()
            
            # Get the word
            predicted_word = tokenizer.index_word[predicted_idx]
            
            # Add to generated text
            generated_text += " " + predicted_word
            
            print(generated_text)
            time.sleep(1)  # Small delay for readability
    
    return generated_text


print("Prediction function defined!")

In [ ]:
# Test the model with different seed texts
test_seeds = [
    "what is the",
    "the course follows",
    "python is for"
]

print("Testing next word prediction:\n")
for seed in test_seeds:
    print(f"\nSeed text: '{seed}'")
    print("\nGenerating next words:")
    print(f"Starting: {seed}")
    result = predict_next_words(seed, model, tokenizer, num_words=5, max_seq_length=5)

In [ ]:
# Save the model
model_save_path = 'next_word_lstm_pytorch.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

# Save tokenizer
import pickle
tokenizer_save_path = 'next_word_tokenizer.pkl'
with open(tokenizer_save_path, 'wb') as f:
    pickle.dump(tokenizer, f)
print(f"Tokenizer saved to {tokenizer_save_path}")

# Save configuration
import json
config = {
    'vocab_size': vocab_size,
    'embedding_dim': embedding_dim,
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'seq_length': seq_length,
    'max_seq_length': max_len
}
config_save_path = 'next_word_config.json'
with open(config_save_path, 'w') as f:
    json.dump(config, f)
print(f"Config saved to {config_save_path}")

In [ ]:
# Function to load and use the saved model
def load_model_inference(model_path, tokenizer_path, config_path, device):
    """
    Load saved model for inference
    """
    # Load config
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    # Load tokenizer
    with open(tokenizer_path, 'rb') as f:
        tokenizer = pickle.load(f)
    
    # Initialize and load model
    model = NextWordLSTM(
        config['vocab_size'],
        config['embedding_dim'],
        config['hidden_size'],
        config['num_layers']
    )
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    
    return model, tokenizer, config


print("Model loading function defined!")
print("\nYou can load the saved model using:")
print(f"model, tokenizer, config = load_model_inference('{model_save_path}', '{tokenizer_save_path}', '{config_save_path}', device)")